# Topic: SQL ROW_NUMBER() Pattern

## Definition (30-second explanation)
* `ROW_NUMBER()` is a window function that assigns a unique, sequential integer to every row within a result set or partition, starting from 1.
* Unlike other ranking functions, it never produces duplicate values; every single row gets its own unique number, regardless of ties.

## Why Interviewers Ask This
* To test your ability to handle "get exactly one row per group" scenarios, which are extremely common in real-world data pipelines.
* To verify you understand the difference between window functions and standard aggregate functions.
* To see if you understand query execution order, specifically that window functions cannot be directly filtered in a `WHERE` clause.

## Core Concepts
* **PARTITION BY:** Divides the result set into groups; `ROW_NUMBER()` resets to 1 at the start of each new partition.
* **ORDER BY (inside OVER):** Defines how rows are ordered within each partition *before* numbers are assigned.
* **CTE / Subquery Requirement:** You must wrap the window function in a Common Table Expression (CTE) or subquery to filter on its result.

## When to Use
* Deduplicating records while keeping the most recent or most important row per group.
* Selecting the top-N rows per group (e.g., top 3 salespeople per region).
* Implementing pagination in APIs or dashboards (e.g., page 1: rows 1-10).
* Finding the first or last event per user/session in event logs.

## Advantages
* Guarantees uniqueness—sets it apart from `RANK()` and `DENSE_RANK()`.
* Perfect for intelligent deduplication and returning deterministic results when tied with a robust `ORDER BY` clause.

## Limitations
* Not suitable when ties should receive the same rank number.
* Can be slow when working with extremely large datasets without proper indexing on the `PARTITION BY` and `ORDER BY` columns.

## Common Comparisons
* **vs RANK():** `RANK()` gives the same number to ties and skips the next ranks (e.g., 1, 1, 3). `ROW_NUMBER()` always gives unique numbers (1, 2, 3).
* **vs DENSE_RANK():** `DENSE_RANK()` gives the same number to ties but does *not* skip the next rank (e.g., 1, 1, 2).
* **vs Correlated Subquery:** Subqueries can achieve similar filtering but often evaluate slower on large datasets and do not elegantly handle tie-breaking for single-row returns.

## Common Interview Traps
* **Forgetting ORDER BY inside OVER():** Without it, row numbers are assigned arbitrarily, leading to non-deterministic results.
* **Using in WHERE directly:** You CANNOT use `ROW_NUMBER()` in a `WHERE` clause directly.
* **Missing PARTITION BY:** Without it, it numbers ALL rows in the entire table instead of resetting per group.
* **Wrong ORDER BY direction:** Forgetting `DESC` when you want the "latest" or "highest" value to get `rn=1`.

## SQL Syntax 
```sql
WITH RankedData AS (
  SELECT 
    column1,
    ROW_NUMBER() OVER (
      PARTITION BY column_to_group_by 
      ORDER BY column_to_sort_by DESC
    ) AS rn
  FROM table_name
)
SELECT * FROM RankedData WHERE rn = 1;
```

## 45-Second Interview Answer
"ROW_NUMBER() is my go-to window function for deduplication and top-N-per-group problems. It assigns a unique, sequential integer to rows within specific partitions, guaranteeing no duplicates even if there are ties. Because window functions evaluate after the WHERE clause, I always implement this pattern by computing the row number inside a CTE, ordering by a tie-breaker like a timestamp, and then querying that CTE to filter where the row number equals 1."

## Example Questions:

### Q1: Find the 3 highest-paid employees in each department. If there are ties, include all tied employees at the boundary.
* **Ideal Answer:** "This is actually a trick question for ROW_NUMBER(). Because the requirement states we must 'include all tied employees at the boundary', ROW_NUMBER() is the wrong tool since it forces unique ranks. I would use DENSE_RANK() instead, partitioned by department and ordered by salary descending, then filter for rank <= 3 in an outer query."
* **Common Mistakes:** Blindly using `ROW_NUMBER()` which would arbitrarily cut off tied employees if there are four people tied for the 3rd highest salary.
* **Follow-up:** "How would your answer change if we strictly only wanted 3 people total per department, regardless of ties?" (Answer: Switch back to ROW_NUMBER and add a secondary order by column, like hire_date, to deterministically break ties).

### Q2: Remove duplicate customer records from a CRM table. Keep the record with the most recent updated_at timestamp.
* **Ideal Answer:** 
```sql
WITH Deduplicated AS (
    SELECT *, 
           ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY updated_at DESC) as rn
    FROM crm_customers
)
-- Depending on if it's a SELECT or a DELETE operation:
SELECT * FROM Deduplicated WHERE rn = 1;
```
* **Common Mistakes:** Forgetting `DESC` in the `ORDER BY` clause, which would keep the oldest record instead of the most recent.
* **Follow-up:** "How does this approach compare to using a GROUP BY with MAX(updated_at)?" (Answer: ROW_NUMBER allows you to easily retrieve *all* other columns associated with that specific timestamp without complex self-joins).

### Q3: Implement pagination: return rows 21 through 30 from a products table sorted by price descending.
* **Ideal Answer:**
```sql
WITH Paginated AS (
    SELECT *, ROW_NUMBER() OVER(ORDER BY price DESC) as rn
    FROM products
)
SELECT * FROM Paginated WHERE rn BETWEEN 21 AND 30;
```
* **Common Mistakes:** Including a `PARTITION BY` when it isn't needed (pagination usually applies to the whole dataset, not grouped subsets).
* **Follow-up:** "Why might OFFSET and FETCH/LIMIT be preferred over ROW_NUMBER() for pagination in some databases?" (Answer: Built-in limit/offset operators can sometimes be heavily optimized by the query planner, whereas ROW_NUMBER() might require sorting the entire dataset first depending on indexes).

### Q4: For each user, find their first and third purchase. Return NULL if they have fewer than 3 purchases.
* **Ideal Answer:**
```sql
WITH NumberedPurchases AS (
    SELECT user_id, purchase_details,
           ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY purchase_date ASC) as rn
    FROM purchases
)
SELECT 
    p1.user_id,
    p1.purchase_details as first_purchase,
    p3.purchase_details as third_purchase
FROM NumberedPurchases p1
LEFT JOIN NumberedPurchases p3 ON p1.user_id = p3.user_id AND p3.rn = 3
WHERE p1.rn = 1;
```
* **Common Mistakes:** Trying to use `WHERE rn IN (1, 3)` in a single query without pivoting, which returns rows instead of columns and fails to explicitly output NULL for the missing 3rd purchase.
* **Follow-up:** "How would you solve this if you needed to pivot 10 different sequential purchases into columns?" (Answer: I would use aggregation with CASE WHEN statements (conditional aggregation) or a PIVOT function if the dialect supports it).

### Q5: Given a clickstream table with user_id, page_url, and timestamp, find the last 2 pages each user visited.
* **Ideal Answer:**
```sql
WITH RankedClicks AS (
    SELECT user_id, page_url, timestamp,
           ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY timestamp DESC) as rn
    FROM clickstream
)
SELECT user_id, page_url, timestamp
FROM RankedClicks
WHERE rn <= 2;
```
* **Common Mistakes:** Forgetting to wrap the window function in a CTE, or using `ASC` instead of `DESC` on the timestamp.
* **Follow-up:** "If a user visits the exact same page twice in a row, how would you ensure you get the last two *distinct* pages they visited?" (Answer: Use `LAG()` to filter out consecutive duplicate URLs before applying the `ROW_NUMBER()`).

## Practice Questions

### Q1:
**Write a SQL query to find the second most recent order for each customer.**

**Constraint: If a customer has only placed one order in their lifetime, they should NOT appear in the final result set. Return the customerNumber, orderNumber, and orderDate.**

**Mock Schema**
```sql
-- Create the orders table
CREATE TABLE orders (
    orderNumber INT PRIMARY KEY,
    customerNumber INT,
    orderDate DATE,
    status VARCHAR(50)
);

-- Insert sample data
INSERT INTO orders (orderNumber, customerNumber, orderDate, status) VALUES
-- Customer 363: Has 3 orders. (2nd most recent is 10101)
(10100, 363, '2003-01-06', 'Shipped'),
(10101, 363, '2003-02-06', 'Shipped'), 
(10102, 363, '2003-03-06', 'Shipped'), 

-- Customer 114: Has 2 orders. (2nd most recent is 10103)
(10103, 114, '2003-01-09', 'Shipped'),
(10104, 114, '2003-01-15', 'Shipped'), 

-- Customer 201: Has only 1 order. (Should NOT appear in final output)
(10105, 201, '2003-01-20', 'Shipped');
```

**Answer:**
```sql
WITH RankedOrders AS (
    SELECT orderNumber, customerNumber, orderDate,
           ROW_NUMBER() OVER(PARTITION BY customerNumber ORDER BY orderDate DESC) as rnk
    FROM orders
)
SELECT customerNumber, orderNumber, orderDate
FROM RankedOrders 
WHERE rnk = 2;
```
* **Interview Tips:** Always verbalize your logic for edge cases. Point out that filtering `WHERE rnk = 2` acts as a natural filter for the constraint, completely eliminating the need for an expensive `HAVING COUNT(*) > 1` clause.

# Q2: (System Design / Optimization) How do you optimize a ROW_NUMBER() OVER(PARTITION BY col1 ORDER BY col2) query on a 500-million row table?
* **Ideal Answer:** "To optimize this window function and avoid a massive file sort, I would add a **Composite Index** on both the partition and order columns: `(customerNumber, orderDate)`. To make it even faster, I would make it a **Covering Index** by including the `orderNumber` column, which allows the query engine to retrieve all necessary data directly from the index without doing a table lookup."
* **SQL Implementation:** `CREATE INDEX idx_customer_date ON orders(customerNumber, orderDate DESC, orderNumber);`
* **Common Mistakes:** Suggesting a single-column index on just the `ORDER BY` column. The database must partition the data first, so the `PARTITION BY` column must be the leading column in the composite index.
* **Follow-up:** "What happens if we also need to pull the `status` column in our SELECT statement?" (Answer: We would either need to add `status` to our covering index, or accept that the engine will have to perform a row lookup for the filtered records).

### Q3:
**Scenario: (The Edge Case Gauntlet)**

**Write a SQL query to find the single most recent successful payment for each user.**

**Business Logic & Edge Cases to Handle:**

- You must only consider payments where status = 'Success'.

- If a user has multiple successful payments on the exact same timestamp, prioritize the one with the highest amount.

- If both the payment_date and the amount are tied, prioritize the smallest transaction_id.

- NULL dates should be treated as the oldest possible payments. (If a user has a real date and a NULL date, the real date wins. If a user only has NULL dates, they should still be included in the results, using the other tie-breakers).

- Return all columns from the payments table.

**Mock Schema**
```sql
CREATE TABLE payments (
    transaction_id INT PRIMARY KEY,
    user_id INT,
    payment_date TIMESTAMP,
    amount DECIMAL(10,2),
    status VARCHAR(20)
);

INSERT INTO payments (transaction_id, user_id, payment_date, amount, status) VALUES
-- User 1: Standard case. (Winner: 102)
(101, 1, '2023-01-01 10:00:00', 50.00, 'Success'),
(102, 1, '2023-01-02 10:00:00', 75.00, 'Success'),

-- User 2: Tie on date. (Winner: 104 because amount is higher)
(103, 2, '2023-01-03 12:00:00', 100.00, 'Success'),
(104, 2, '2023-01-03 12:00:00', 150.00, 'Success'),

-- User 3: Tie on date AND amount. (Winner: 105 because ID is smaller)
(106, 3, '2023-01-04 15:00:00', 200.00, 'Success'),
(105, 3, '2023-01-04 15:00:00', 200.00, 'Success'),

-- User 4: Only Failed payments. (Should NOT appear in final output)
(107, 4, '2023-01-05 10:00:00', 300.00, 'Failed'),

-- User 5: Has a NULL date. (Winner: 109 because known date is better than NULL)
(108, 5, NULL, 50.00, 'Success'),
(109, 5, '2023-01-06 09:00:00', 40.00, 'Success'),

-- User 6: ONLY Null dates. (Winner: 110 based on amount tie-breaker)
(110, 6, NULL, 500.00, 'Success'),
(111, 6, NULL, 250.00, 'Success');
```

* **Answer:**
```sql
WITH RankedPayments AS (
    SELECT transaction_id, user_id, payment_date, amount, status,
           ROW_NUMBER() OVER(
               PARTITION BY user_id 
               ORDER BY payment_date DESC, amount DESC, transaction_id ASC
           ) as rnk
    FROM payments
    WHERE status = 'Success'
)
SELECT transaction_id, user_id, payment_date, amount, status
FROM RankedPayments
WHERE rnk = 1;
```
* **Interview Tips:** When faced with complex tie-breakers, stack them sequentially inside the `ORDER BY` of the window function. Always filter out invalid rows (like failed payments) *inside* the CTE before the window function evaluates to save compute resources.
* **Common Mistakes:** Forgetting that different SQL dialects handle `NULL` sorting differently. In MySQL and SQL Server, `ORDER BY col DESC` puts `NULL`s last. In PostgreSQL and Oracle, `ORDER BY col DESC` puts `NULL`s first, requiring an explicit `NULLS LAST` modifier.
* **Follow-up:** "How would the performance change if we moved the `WHERE status = 'Success'` filter to the outer query instead of the CTE?" (Answer: It would perform worse because the database would waste time computing row numbers for failed transactions before throwing them away.)